In [1]:
# %% [markdown]
# ## 2.1 Imports

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
)
import matplotlib.pyplot as plt

In [2]:
# %% [markdown]
# ## 2.2 Load real CSV or generate a synthetic dataset
#
# Option A: load from CSV if you have it
#
# file_path = Path("tabio_uk_sales_demographics.csv")
# df = pd.read_csv(file_path)
#
# Uncomment above and ensure the CSV has columns matching the schema.
#
# Option B: create a synthetic dataset for experimentation
#

np.random.seed(42)

# synthetic size
n_samples = 5000

# demographics
age_groups = ["18-24", "25-34", "35-44", "45-54", "55+"]
genders = ["Male", "Female", "Other"]
income_bands = ["Low", "Medium", "High"]

# regions (example UK regions; adjust to your real data)
regions = ["London", "South East", "Midlands", "North West", "Scotland", "Wales"]

# products and categories
products = [f"prod_{i}" for i in range(1, 21)]
categories = ["Socks", "Tights", "Legwear", "Accessories"]

df = pd.DataFrame({
    "sale_id": range(1, n_samples + 1),
    "date": pd.to_datetime("2024-01-01") + pd.to_timedelta(
        np.random.randint(0, 365, size=n_samples), unit="d"
    ),
    "store_id": np.random.choice(["Store_A", "Store_B", "Online"], size=n_samples, p=[0.3, 0.2, 0.5]),
    "region": np.random.choice(regions, size=n_samples),
    "product_id": np.random.choice(products, size=n_samples),
    "category": np.random.choice(categories, size=n_samples),
    "units_sold": np.random.poisson(lam=2, size=n_samples).clip(min=1),
})

# set prices, cost, revenue, profit
# base price by category
price_map = {"Socks": 8.0, "Tights": 12.0, "Legwear": 10.0, "Accessories": 15.0}
cost_ratio_map = {"Socks": 0.45, "Tights": 0.5, "Legwear": 0.5, "Accessories": 0.55}

df["price"] = df["category"].map(price_map)
df["cost_ratio"] = df["category"].map(cost_ratio_map)

# add randomness
df["revenue"] = df["price"] * df["units_sold"] * np.random.uniform(0.9, 1.2, size=n_samples)
df["cost"] = df["revenue"] * df["cost_ratio"] * np.random.uniform(0.9, 1.1, size=n_samples)

df["profit"] = df["revenue"] - df["cost"]
df["profit_margin"] = df["profit"] / df["revenue"]

# demographics
df["age_group"] = np.random.choice(age_groups, size=n_samples)
df["gender"] = np.random.choice(genders, size=n_samples, p=[0.48, 0.48, 0.04])
df["income_band"] = np.random.choice(income_bands, size=n_samples, p=[0.4, 0.4, 0.2])

df.head()

,sale_id,date,store_id,region,product_id,category,units_sold,price,cost_ratio,revenue,cost,profit,profit_margin,age_group,gender,income_band
0,1,2024-04-12,Store_A,London,prod_10,Socks,2,8.0,0.45,19.005633,8.803133,10.202500,0.536815,35-44,Other,Low
1,2,2024-12-14,Store_A,South East,prod_7,Accessories,3,15.0,0.55,48.154495,26.085439,22.069056,0.458297,18-24,Female,Medium
2,3,2024-09-27,Store_A,North West,prod_19,Tights,1,12.0,0.50,13.256175,6.882138,6.374037,0.480835,18-24,Other,Low
3,4,2024-04-16,Store_B,Scotland,prod_19,Socks,1,8.0,0.45,9.263600,4.374285,4.889316,0.527799,55+,Female,Medium
4,5,2024-03-12,Online,Wales,prod_19,Tights,1,12.0,0.50,12.149318,6.418560,5.730759,0.471694,55+,Male,Low


In [3]:
# %% [markdown]
# ## 2.3 Define target based on profit margin
# Choose threshold, e.g., 30% margin
threshold = 0.30
df["high_margin"] = (df["profit_margin"] >= threshold).astype(int)

# Inspect distribution
print("High margin share:", df["high_margin"].mean())

High margin share: 1.0


In [4]:
# %% [markdown]
# ## 2.4 Feature selection and train/test split
#
# Use demographic and a few sales-related numeric features
features = [
    "region",
    "category",
    "store_id",
    "age_group",
    "gender",
    "income_band",
    "units_sold",
    "revenue",
    # optionally other features
]

X = df[features]
y = df["high_margin"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [6]:
# %% [markdown]
# ## 2.5 Preprocessing pipelines
# Categorical columns: one-hot encode
# Numeric columns: scale

categorical_cols = [
    "region",
    "category",
    "store_id",
    "age_group",
    "gender",
    "income_band",
]

numeric_cols = ["units_sold", "revenue"]

# transformers
cat_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
num_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", cat_transformer, categorical_cols),
        ("num", num_transformer, numeric_cols),
    ]
)

In [10]:
print(y.unique())
print(y.value_counts())
print(y_train.value_counts())
print(y_test.value_counts())

[1]
high_margin
1    5000
Name: count, dtype: int64
high_margin
1    4000
Name: count, dtype: int64
high_margin
1    1000
Name: count, dtype: int64


In [12]:
y.value_counts()

high_margin
1    5000
Name: count, dtype: int64

In [22]:
print(df.head())
print(df.columns)
print(df.shape)

Empty DataFrame
Columns: [id, diagnosis, radius_mean, texture_mean, perimeter_mean, area_mean, smoothness_mean, compactness_mean, concavity_mean, concave points_mean, symmetry_mean, fractal_dimension_mean, radius_se, texture_se, perimeter_se, area_se, smoothness_se, compactness_se, concavity_se, concave points_se, symmetry_se, fractal_dimension_se, radius_worst, texture_worst, perimeter_worst, area_worst, smoothness_worst, compactness_worst, concavity_worst, concave points_worst, symmetry_worst, fractal_dimension_worst, Unnamed: 32]
Index: []

[0 rows x 33 columns]
Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', '

In [23]:
# %% [markdown]
# ## 2.9 Save the processed dataset or model if desired
#
# Example: save the training subset to a CSV for inspection
X_train_with_target = X_train.copy()
X_train_with_target["high_margin"] = y_train
X_train_with_target.to_csv("tabio_uk_train_sample.csv", index=False)

# Example: save model with joblib (install joblib if needed)
# from joblib import dump
# dump(model, "tabio_uk_high_margin_model.joblib")